In [ ]:
from pathlib import Path
import pandas as pd

from src.cilia_detection.metrics import segmentation_metrics, detection_metrics, calc_precision_recall
from src.cilia_detection.utils import yolobbox2bbox
import cv2

In [ ]:
TYPE = "difficult"
proj_dir = Path(f"../../data/cilia_dataset/{TYPE}")

# Detection metrics

In [3]:
gt_dir = proj_dir / "results_cellprofiler/bboxes"
pred_dir = proj_dir / "results_cilia/bboxes"

In [4]:
img_results = {}
for p in gt_dir.glob("*.txt"):
    with open(p, 'r') as f:
        lines = f.readlines()
        gt_bboxes = [yolobbox2bbox([float(p) for p in line.split(' ')[1:]]) for line in lines]
    with open(pred_dir / p.name, 'r') as f:
        lines = f.readlines()
        pred_bboxes = [yolobbox2bbox([float(p) for p in line.split(' ')[1:]]) for line in lines]

    img_results[p.stem] = detection_metrics(gt_bboxes, pred_bboxes, 0.5)
p, r, f1 = calc_precision_recall(img_results)
print('Easy cases:')
print('\tPrecision:', p, 'Recall:', r, 'F1-score:', f1)

Easy cases:
	Precision: 1.0 Recall: 1.0 F1-score: 0.99999950000025


In [5]:
metrics = pd.DataFrame(img_results).transpose().sort_index().round(3)
metrics.to_csv(pred_dir.parent / "detection_metrics.csv")
metrics

,true_pos,false_pos,false_neg,accuracy,precision,recall,f1
"Nthyori,exp1,cond1,image10",9.0,0.0,0.0,1.0,1.0,1.0,1.0
"Nthyori,exp1,cond1,image2",7.0,0.0,0.0,1.0,1.0,1.0,1.0
"Nthyori,exp1,cond1,image3",10.0,0.0,0.0,1.0,1.0,1.0,1.0
"Nthyori,exp1,cond1,image4",9.0,0.0,0.0,1.0,1.0,1.0,1.0
"Nthyori,exp1,cond1,image6",21.0,0.0,0.0,1.0,1.0,1.0,1.0


# Segmentation metrics

In [6]:
gt_dir = proj_dir / "results_cellprofiler/labels"
pred_dir = proj_dir / "results_cilia/labels"

img_results = {}
for p in gt_dir.glob("*.png"):
    gt_mask = cv2.imread(p.as_posix(), cv2.IMREAD_GRAYSCALE)
    pred_mask = cv2.imread((pred_dir / p.name).as_posix(), cv2.IMREAD_GRAYSCALE)
    img_results[p.stem] = segmentation_metrics(gt_mask, pred_mask)

In [7]:
seg_metrics = pd.DataFrame(img_results).transpose().sort_index().round(3)
seg_metrics.to_csv(pred_dir.parent / "segmentation_metrics.csv")
seg_metrics

,accuracy,precision,recall,f1,iou,boundary_f1,hausdorff_distance,specificity,fpr
"Nthyori,exp1,cond1,image10",1.000,0.970,0.944,0.957,0.918,0.459,3.606,1.000,0.000
"Nthyori,exp1,cond1,image2",1.000,0.946,0.991,0.968,0.938,0.560,2.000,1.000,0.000
"Nthyori,exp1,cond1,image3",1.000,0.961,0.974,0.968,0.937,0.545,3.000,1.000,0.000
"Nthyori,exp1,cond1,image4",1.000,0.947,0.967,0.957,0.917,0.639,17.088,1.000,0.000
"Nthyori,exp1,cond1,image6",0.999,0.801,1.000,0.889,0.801,0.023,9.220,0.999,0.001
